# Notebook 05 — Condition D: LSTM with Missingness-Aware Preprocessing

## Requires
- `data/raw/` (training_setA and training_setB from PhysioNet 2019)
- `data/splits/train_ids.csv`, `val_ids.csv`, `test_ids.csv` (from notebook 01)

## Produces
- `results/models/lstm_condition_D.pt`
- Row appended to `results/experiment_log.csv`

## Run Order
Run **AFTER** notebook 01.

> **GPU required:** Before running, enable GPU runtime.
> - Google Colab: Runtime > Change runtime type > T4 GPU
> - Kaggle: Settings > Accelerator > GPU P100

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import torch

from src.config import (
    RANDOM_SEED, DATA_DIR, PROCESSED_DIR, SPLITS_DIR,
    MODELS_DIR, METRICS_DIR, EXPERIMENT_LOG, MAX_SEQ_LEN,
    ALL_FEATURES, VITAL_SIGNS, LAB_VALUES, OUTLIER_BOUNDS,
    LABEL_SHIFT_HOURS, SPLIT_RATIOS,
)
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers, apply_strategy_B
from src.utils import set_all_seeds, validate_no_nans, create_patient_splits
from src.models import SepsisLSTM
from src.train import make_loaders, train_lstm
from src.evaluate import select_threshold, compute_all_metrics, log_results

set_all_seeds(RANDOM_SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print('Imports OK')

## 1. Load Data and Create Splits

In [ ]:
# ── Load and preprocess ───────────────────────────────────────────────────
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df = clip_outliers(df)

# Patient-level stratified split (same seed as all other conditions)
# create_patient_splits uses SPLIT_RATIOS and RANDOM_SEED from config internally
patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df["patient_id"].isin(patient_train)].copy()
val_df   = df[df["patient_id"].isin(patient_val)].copy()
test_df  = df[df["patient_id"].isin(patient_test)].copy()

print(f"Train patients: {len(patient_train):,} | Val: {len(patient_val):,} | Test: {len(patient_test):,}")

## 2. Strategy B Preprocessing (forward-fill + missingness indicators)

In [ ]:
# ── Strategy B: forward-fill + missingness indicators + StandardScaler ───
# Returns 76 features: 40 original + 36 missingness indicator columns
X_train, X_val, X_test, y_train, y_val, y_test, feature_cols_B = apply_strategy_B(
    train_df, val_df, test_df
)

# Reassemble into DataFrames for SepsisDataset.
# IMPORTANT: Strategy B explicitly excludes ICULOS from feature_cols_B (it is in
# META_COLS in preprocessing.py). SepsisDataset needs ICULOS to sort timesteps,
# so we add it back from the original DataFrame as unscaled metadata.
def make_scaled_df(original_df, X_scaled, feat_cols):
    meta = original_df[["patient_id", "ICULOS", "EarlyLabel"]].reset_index(drop=True)
    feat = pd.DataFrame(X_scaled, columns=feat_cols)
    return pd.concat([meta, feat], axis=1)

train_lstm_df = make_scaled_df(train_df, X_train, feature_cols_B)
val_lstm_df   = make_scaled_df(val_df,   X_val,   feature_cols_B)
test_lstm_df  = make_scaled_df(test_df,  X_test,  feature_cols_B)

print(f"Device: {device}  Features: {len(feature_cols_B)}")
print(f"(Condition C had 40 features; Condition D has {len(feature_cols_B)} — adds missingness indicators)")
assert "ICULOS" in train_lstm_df.columns, "ICULOS missing — SepsisDataset will crash"
print("ICULOS present: OK")

## 3. Hyperparameter Grid Search

In [ ]:
# ── Class imbalance weight ───────────────────────────────────────────────
pos_count = int(y_train.sum())
neg_count = int(len(y_train) - pos_count)
pos_weight = neg_count / pos_count
print(f'pos_weight = {pos_weight:.1f}  (neg={neg_count:,} / pos={pos_count:,})')

# ── Same grid as Condition C — ensures fair C vs D comparison ────────────
param_grid = [
    {'hidden_size': h, 'dropout': d, 'lr': lr}
    for h   in [64, 128]
    for d   in [0.2, 0.3]
    for lr  in [1e-3, 5e-4]
]

best_val_auprc = 0.0
best_params    = None
best_model     = None

for params in param_grid:
    print(f"\n--- hidden={params['hidden_size']}, dropout={params['dropout']}, lr={params['lr']} ---")
    set_all_seeds(RANDOM_SEED)

    train_loader, val_loader, test_loader = make_loaders(
        train_lstm_df, val_lstm_df, test_lstm_df,
        patient_train, patient_val, patient_test,
        feature_cols=feature_cols_B, batch_size=64,
    )

    model = SepsisLSTM(
        input_size=len(feature_cols_B),   # 76 for Strategy B
        hidden_size=params['hidden_size'],
        num_layers=2,
        dropout=params['dropout'],
    )

    model, val_auprc = train_lstm(
        model, train_loader, val_loader,
        n_epochs=50, lr=params['lr'],
        pos_weight=pos_weight,
        patience=10, device=device,
    )

    print(f"Result: val AUPRC={val_auprc:.4f}")
    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        best_params    = params
        best_model     = model

print(f'\nBest: {best_params}  val AUPRC: {best_val_auprc:.4f}')

## 4. Evaluate Best Model

In [ ]:
# ── Evaluate best model on val and test ──────────────────────────────────
def get_probs_labels(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            logits = model(X_batch.to(device), lengths)
            probs  = torch.sigmoid(logits)
            seq_len = logits.shape[1]
            mask = torch.zeros(y_batch.shape[0], seq_len, dtype=torch.bool)
            for i, l in enumerate(lengths):
                mask[i, :min(int(l), seq_len)] = True
            all_probs.extend(probs[mask].cpu().numpy())
            all_labels.extend(y_batch[:, :seq_len][mask].numpy())
    return np.array(all_probs), np.array(all_labels)

# Rebuild loaders with best params for clean evaluation
train_loader, val_loader, test_loader = make_loaders(
    train_lstm_df, val_lstm_df, test_lstm_df,
    patient_train, patient_val, patient_test,
    feature_cols=feature_cols_B, batch_size=64,
)

val_probs,  val_labels  = get_probs_labels(best_model, val_loader,  device)
test_probs, test_labels = get_probs_labels(best_model, test_loader, device)

threshold    = select_threshold(val_labels, val_probs)
val_metrics  = compute_all_metrics(val_labels,  val_probs,  threshold)
test_metrics = compute_all_metrics(test_labels, test_probs, threshold)

print('\n=== CONDITION D RESULTS ===')
print(f'Test AUC-ROC : {test_metrics["auc_roc"]:.4f}')
print(f'Test AUPRC   : {test_metrics["auprc"]:.4f}')
print(f'Test F1      : {test_metrics["f1"]:.4f}')

log_results(
    condition='D',
    model='LSTM',
    strategy='Strategy_B',
    val_metrics=val_metrics,
    test_metrics=test_metrics,
    hyperparams=best_params,
)

## 5. Save Model

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
torch.save(best_model.state_dict(), f'{MODELS_DIR}lstm_condition_D.pt')
joblib.dump(feature_cols_B, f'{MODELS_DIR}lstm_condition_D_feature_cols.pkl')
print(f'Model saved to {MODELS_DIR}lstm_condition_D.pt')

## 6. Interpreting the Key Comparison: Condition B vs D

This comparison answers the central question of the project:
**does temporal modeling (LSTM) add value when preprocessing is already optimal (Strategy B)?**

| Condition | Model | Strategy | Test AUPRC |
|---|---|---|---|
| B | XGBoost | Strategy B (forward-fill + indicators) | 0.1087 |
| D | LSTM    | Strategy B (forward-fill + indicators) | *see above* |

**Interpretation guide:**

- **D > B**: The LSTM extracts temporal patterns that XGBoost misses even with the same features — temporal *architecture* matters independently of preprocessing.
- **D ≈ B**: Once missingness structure is captured explicitly, the LSTM gains no further advantage — *preprocessing design is the more important lever* (confirms our central hypothesis).
- **D < B**: The LSTM struggles even with better features, suggesting the training signal is insufficient for the sequence model at this dataset size.

Also compare **C vs D** (both LSTM, different strategy):
- **D > C** confirms that missingness indicators help the LSTM learn better representations.